<a href="https://colab.research.google.com/github/roy-tokyo/aiagent/blob/main/aiagent_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#01-OpenAI APIを使用する簡単なアプリケーションを作る

openaiライブラリを使用するシンプルな例です。

In [ ]:
from openai import OpenAI
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI();

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role":"system","content":"You are a helpful assistant."},
        {"role":"user","content":"狛江はどこにあるかを教えて！"}
    ],
)

print(response.to_json(indent=2))

{
  "id": "chatcmpl-DPfyhbnKlAtA1cUmuLq5PimI8g0Bm",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "狛江（こまえ）は、日本の東京都の西部に位置する市です。狛江市は、東京都の多摩地域に属しており、周囲には調布市や世田谷区などがあります。狛江は、小田急小田原線の沿線にあり、都心へのアクセスも良好です。市は緑が豊かで、住みやすい場所として知られています。",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1775011935,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_f85b8886b6",
  "usage": {
    "completion_tokens": 96,
    "prompt_tokens": 30,
    "total_tokens": 126,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


同じ例でストリーミングで解答を得られるようにします。

In [3]:
from openai import OpenAI
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI();response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role":"system","content":"You are a helpful assistant."},
        {"role":"user","content":"村上 宗隆選手はどの球団に所属するか"}
    ],
    stream = True
)

for chunk in response:
  content = chunk.choices[0].delta.content
  if content is not None:
    print(content, end="",flush=True)

村上 宗隆選手は、東京ヤクルトスワローズに所属しています。彼はプロ野球選手で、主に一塁手としてプレーしています。

In [2]:
from openai import OpenAI
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI();

response = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role":"system","content":'地名を次のJSON形式で出力してください。¥n{"地名":["aaa","bbb"]}'},
        {"role":"user","content":"東京の23区を教えて！"}
    ],
    #jsonで応答するように指定する
    response_format={"type":"json_object"}
)

print(response.choices[0].message.content)

{"地名":["千代田区","中央区","港区","新宿区","文京区","台東区","墨田区","江東区","品川区","目黒区","大田区","世田谷区","渋谷区","中野区","杉並区","豊島区","北区","荒川区","板橋区","練馬区","足立区","葛飾区","江戸川区"]}


#02-LangChainによるAgent開発


In [6]:
# langsmith、langchainをインストール
!pip install -q langsmith langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 9.9 MB/s eta 0:00:00


In [ ]:
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# LangSmith Clientを作成
client = Client()

# プロンプトを取得
prompt = client.pull_prompt("rlm/rag-prompt")
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [ ]:
from langsmith import Client

# LangChain Hub  からReActプロンプト取得
client = Client()
prompt = client.pull_prompt("hwchase17/react")

print("=== ReActプロンプト ===")
print(prompt)

=== ReActプロンプト ===
input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [5]:
!pip install -q langchain-community google-search-results --break-system-packages

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


SerpAPIWrapper1の使い方を確認する。

In [ ]:
from langchain_community.utilities import SerpAPIWrapper
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")

search = SerpAPIWrapper()
result = search.run("Python programming")
print(result)

['The official home of the Python Programming Language.', 'Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation.', 'Python is a popular programming language. It was created by Guido van Rossum, and released in 1991. It is used for:', 'Learn Python programming with AI tools. This course takes you from fundamentals to best practices using Codex CLI for serious development.', 'Python is a computer programming language often used to build websites and software, automate tasks, and conduct data analysis.']


In [ ]:
!pip install langsmith --break-system-packages

# 03‐ReAct Agent とは

**ReAct** = **Reasoning** (推論) + **Acting** (行動)

LLM（大規模言語モデル）が「考えながら行動する」ためのフレームワークです。

## 基本的な動作フロー

1. **Question（質問）**: ユーザーからの質問を受け取る
2. **Thought（思考）**: 「何をすべきか」を考える
3. **Action（行動）**: ツール（検索、計算など）を使って情報を取得
4. **Observation（観察）**: ツールの実行結果を確認
5. **Thought → Action → Observation を繰り返す**
6. **Final Answer（最終回答）**: 十分な情報が集まったら回答

## 例
```
Question: 2024年のノーベル物理学賞受賞者は？

Thought: 最新情報が必要なので検索ツールを使おう
Action: Search
Action Input: "2024年 ノーベル物理学賞"
Observation: ジョン・ホップフィールドとジェフリー・ヒントンが受賞

Thought: 情報が得られたので回答できる
Final Answer: 2024年のノーベル物理学賞は、ジョン・ホップフィールドと
ジェフリー・ヒントンが受賞しました。
```

## メリット

✅ **透明性**: 思考プロセスが見える  
✅ **柔軟性**: 複数のツールを組み合わせて使える  
✅ **信頼性**: 推論の根拠が明確

LangGraphのcreate_react_agentを使って、ReAct型のAgentを作って見た。

In [11]:
from langchain_core.tools import Tool
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import ChatOpenAI
from langsmith import Client
from langgraph.prebuilt import create_react_agent
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")
# LLMの設定
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# ツールの設定
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events"
    )
]

# LangSmithクライアントでReActプロンプトを取得
client = Client()
#prompt_template = client.pull_prompt("langchain-ai/react-agent-template")
prompt_template = client.pull_prompt("hwchase17/react")

print("=== ReActプロンプト ===")
print(prompt_template)


# プロンプトテキストを抽出してシステムプロンプトとして使用
system_prompt = prompt_template.template if hasattr(prompt_template, 'template') else str(prompt_template)

# LangGraphでReActエージェントを作成（promptパラメータを使用）
agent_executor = create_react_agent(
    llm,
    tools,
    prompt=system_prompt  # 文字列として渡す
)

# 実行
#question = "Agent的最新研究成果是什么？"
#question = "2026/4/1ドジャーズの先発投手は？"
question = "村上 宗隆選手は現在どの球団に所属するか"
result = agent_executor.invoke({
    "messages": [("user", question)]
})

# 結果の表示
print("\n=== Agent Response ===")
for message in result["messages"]:
    print(f"{message.type}: {message.content}")
    print("---")

=== ReActプロンプト ===
input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


/tmp/ipykernel_2082/2569807585.py:37: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(



=== Agent Response ===
human: 村上 宗隆選手は現在どの球団に所属するか
---
ai: 
---
tool: ['2025年オフにポスティングシステムを利用してMLB球団へ移籍することを表明、2026年よりMLBのホワイトソックスでプレーする。 愛称はムネ、村神様。後者は2022年 ...', '村上 宗隆; むらかみ・むねたか. 投打, 右投左打. 身長／体重, 188cm／97kg. 生年 ... 所属球団, 試合, 打席, 打数, 得点, 安打, 二塁打, 三塁打, 本塁打, 塁打, 打点, 盗塁 ...', '2021年東京五輪、2023年WBCでは日本代表の優勝に貢献。ポスティングシステムを利用し、米大リーグのホワイトソックスと2年総額3400万ドル（約53億4000万円）で契約した。背 ...', 'ホワイトソックスが（中略）メジャー移籍を目指していた村上宗隆内野手（25）と契約合意したと発表した。 · ホワイトソックスとは－。 · 高津臣吾氏（57）が（中略） ...', '【シカゴ=共同】プロ野球ヤクルトからポスティングシステムで米大リーグのホワイトソックスに移籍した村上宗隆内野手（25）が22日、シカゴの本拠地 ...', 'ポスティング・システムにより、村上宗隆選手が米大リーグ「シカゴ・ホワイトソックス」との契約が成立したことを、お知らせいたします。 村上宗隆選手 ...', '... 村上宗隆#東京ヤクルトスワローズ #ポスティング#移籍 # ... 【MLB移籍】岡本と今井は強豪チームの要になれるのか？！2026年ア ...', 'ポスティング・システムによりMLB移籍を目指していたヤクルト・村上宗隆内野手がシカゴ・ホワイトソックスと契約合意。今年までヤクルトの監督だった ...', '村上 宗隆 むらかみ むねたか ; 九州学院高 ; 東京ヤクルトスワローズ ; シカゴ・ホワイトソックス ...', 'MLB シカゴ・ホワイトソックス 村上 宗隆のプロフィール、個人成績をお届けします。']
---
tool: ['村上 宗隆（むらかみ むねたか、2000年2月2日 - ）は、熊本県熊本市東区出身のプロ野球選手（内野手）。右投左打。MLBのシカゴ・ホワイトソックス所属。', '投打, 右投左打. 身長／体重, 

同じく質問「村上 宗隆選手は現在どの球団に所属するか」を聞くと、検索ツール有無のよって、解答の精度が全然違います。

No Tool

村上 宗隆選手は、東京ヤクルトスワローズに所属しています。彼はプロ野球選手で、主に一塁手としてプレーしています。

Tool

村上 宗隆選手は現在MLBのシカゴ・ホワイトソックスに所属しています。